# Part 1: Learn the language of Clifford + T

In [1]:
from typing import Any

from kirin.ir import Method
import matplotlib.pyplot as plt
import numpy as np

from bloqade import squin, tsim, stim
from bloqade.cirq_utils import emit_circuit, load_circuit, noise
from bloqade.pyqrack import StackMemorySimulator
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
import matplotlib.pyplot as plt

# this will help us have return types for our methods that have more intuitive names
Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

# this function will help us visualize some circuits
def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()

    return tsim.Circuit(_to_visualize).diagram(height=400)

# Part 2: Synthesize the rotation family

In [49]:
theta = np.pi/8
gate_sequence = ['S', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'S', 'T', 'H', 'S', 'H', 'T', 'H', 'S', 'T', 'H', 'S', 'T', 'H', 'S', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'S', 'H', 'S', 'H', 'S', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'S', 'H', 'S', 'H', 'T', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'T']

@squin.kernel
def rotation_circuit():
    qubits = squin.qalloc(1)
    for gate in ['S', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'S', 'T', 'H', 'S', 'H', 'T', 'H', 'S', 'T', 'H', 'S', 'T', 'H', 'S', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'S', 'H', 'S', 'H', 'S', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'S', 'H', 'S', 'H', 'T', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'T']:
        if gate == 'H':
            squin.h(qubits[0])
        elif gate == 'S':
            squin.s(qubits[0])
        elif gate == 'T':
            squin.t(qubits[0])

    #Inversa de teorica
    squin.rz(-theta, qubits[0])
    m = squin.broadcast.measure(qubits)
    return m

#show_circuit(rotation_circuit)
pyqrack_target = StackMemorySimulator(min_qubits=1)
task = pyqrack_target.task(rotation_circuit)
batch_results = task.batch_run(shots=100)
batch_results

{(<Measurement.Zero: 0>,): 1.0}

Cuando este ya todo, runear los casos n=4 y n=5

# Part 3: Non-Clifford gates are expensive

The optimal way is through a teleportation algorithm.
Parameters:
- Number of ancilla qubits = Nº of T in the 1 qubit circuit.
- Number of 2-qubit gates = Nº of CNOTs = Nº of ancilla qubits.
- Circuit depth = Each additional T gate adds a parallel subroutine that needs to be done. In the operations in the main qubit, only 1 2-qubit-gate is added per T. But if we now consider the process to create the |T> magic state, then the complexity is greater, since T gates are not fault tolerant normally. They should be created via purification.
- Feed forward: the more T you employ, you start to be limited by the time to measure the ancilla, which may dominate over the coherence time of the Rydberg atoms.


Circuito que funciona

In [57]:
@squin.kernel
def t_factory(register: Register):
    squin.reset(register[1])
    squin.h(register[1])
    squin.t(register[1])
    squin.cx(register[0], register[1])
    m = squin.measure(register[1])
    if m == 1:
        squin.s(register[0])
    
@squin.kernel
def generated():
    qubits = squin.qalloc(2)
    for gate in ['H', 'S', 'T']:

        if gate == 'H':
            squin.h(qubits[0])

        elif gate == 'S':
            squin.s(qubits[0])

        elif gate == 'T':
            t_factory(qubits)
        
    m = squin.broadcast.measure(qubits)
    return m[0]

pyqrack_target = StackMemorySimulator(min_qubits=2)
task = pyqrack_target.task(generated)

single_shot = task.run()
single_shot


<Measurement.One: 1>

Benchmarking the T-injection

In [63]:
@squin.kernel
def t_factory_benchmark():
    register = squin.qalloc(2)
    squin.reset(register[1])
    squin.h(register[1])
    squin.t(register[1])
    squin.cx(register[0], register[1])
    m = squin.measure(register[1])
    if m == 1:
        squin.s(register[0])

    #Inversa: agregar 7 T's
    for _ in range(7):
        squin.t(register[0])
    
    m2 = squin.measure(register[0])
    return m2

pyqrack_target = StackMemorySimulator(min_qubits=2)
task = pyqrack_target.task(t_factory_benchmark)
batch_results = task.batch_run(shots=1000)
batch_results

{<Measurement.Zero: 0>: 1.0}

Benchmarking with Rz rotation

In [65]:
theta = np.pi/8
gate_sequence = ['S', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'S', 'T', 'H', 'S', 'H', 'T', 'H', 'S', 'T', 'H', 'S', 'T', 'H', 'S', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'T', 'H', 'T', 'H', 'S', 'H', 'S', 'H', 'S', 'H', 'S', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'T', 'H', 'S', 'T', 'H', 'T', 'H', 'S', 'H', 'S', 'H', 'T', 'H', 'S', 'T', 'H', 'S', 'H', 'S', 'T', 'H', 'S', 'T']

@squin.kernel
def t_factory(register: Register):
    squin.reset(register[1])
    squin.h(register[1])
    squin.t(register[1])
    squin.cx(register[0], register[1])
    m = squin.measure(register[1])
    if m == 1:
        squin.s(register[0])

@squin.kernel
def generated_benchmark():
    qubits = squin.qalloc(2)
    for gate in gate_sequence:

        if gate == 'H':
            squin.h(qubits[0])

        elif gate == 'S':
            squin.s(qubits[0])

        elif gate == 'T':
            t_factory(qubits)
    #Inversa de teorica
    squin.rz(-theta, qubits[0])
    m = squin.broadcast.measure(qubits)
    return m[0]

pyqrack_target = StackMemorySimulator(min_qubits=2)
task = pyqrack_target.task(generated_benchmark)
batch_results = task.batch_run(shots=100)
batch_results

{<Measurement.Zero: 0>: 1.0}

# Part 4: Move from one physical qubit to one logical qubit